In [ ]:
#DailyChallenge

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("="*70)
print("CLASSIFICATION DU CANCER DU SEIN - COMPARAISON DE 4 MODELES")
print("="*70)

# ============================================================================
# 1. ANALYSE EXPLORATOIRE DES DONNEES (EDA)
# ============================================================================

print("\n1. ANALYSE EXPLORATOIRE DES DONNEES")
print("-"*50)

df = pd.read_csv('data.csv')

print(f"Jeu de donnees charge : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print("\nPremieres lignes du dataset :")
print(df.head())

print("\nInformations sur les colonnes :")
print(df.info())

print("\nStatistiques descriptives :")
print(df.describe().round(2))

print("\nVerification des valeurs manquantes :")
print(df.isnull().sum().sum())
print("Aucune valeur manquante detectee")

colonne_id = 'id'
if colonne_id in df.columns:
    print(f"\nSuppression de la colonne '{colonne_id}' (non pertinente pour la modelisation)")
    df = df.drop(columns=[colonne_id])

colonne_sans_nom = 'Unnamed: 32'
if colonne_sans_nom in df.columns:
    print(f"Suppression de la colonne '{colonne_sans_nom}'")
    df = df.drop(columns=[colonne_sans_nom])

print(f"\nDimensions apres suppression : {df.shape}")

print("\nDistribution de la variable cible 'diagnosis' :")
diagnosis_counts = df['diagnosis'].value_counts()
for label, count in diagnosis_counts.items():
    nom_complet = "Malin (Malignant)" if label == 'M' else "Benin (Benign)"
    print(f"   {label} ({nom_complet}) : {count} patients ({count/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
colors = ['#2ECC71', '#E74C3C']
bars = ax1.bar(['Benin (B)', 'Malin (M)'], diagnosis_counts.values, color=colors, edgecolor='black')
ax1.set_ylabel('Nombre de patients', fontsize=12)
ax1.set_title('Distribution des diagnostics', fontsize=14, fontweight='bold')
for bar, count in zip(bars, diagnosis_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            f'{count}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2 = axes[1]
ax2.pie(diagnosis_counts.values, labels=['Benin (B)', 'Malin (M)'], 
        autopct='%1.1f%%', colors=colors, startangle=90, explode=(0.05, 0.05))
ax2.set_title('Proportion des classes', fontsize=14, fontweight='bold')

plt.suptitle('Analyse de la variable cible', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

feature_cols = [col for col in df.columns if col != 'diagnosis']
print(f"\nNombre de features : {len(feature_cols)}")
print("Types de features :")
print("   - mean_* : moyenne des caracteristiques")
print("   - _se_* : erreur standard")
print("   - _worst_* : pire valeur (moyenne des 3 plus grandes)")

fig, axes = plt.subplots(2, 5, figsize=(16, 10))
axes = axes.flatten()

mean_cols = [col for col in feature_cols if col.startswith('mean')][:10]

for i, col in enumerate(mean_cols):
    ax = axes[i]
    for diagnosis, color in zip(['B', 'M'], ['#2ECC71', '#E74C3C']):
        subset = df[df['diagnosis'] == diagnosis][col]
        ax.hist(subset, bins=20, alpha=0.6, label=f'{diagnosis}', color=color, density=True)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Densite', fontsize=9)
    ax.set_title(f'Distribution de {col}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

for i in range(len(mean_cols), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Distribution des principales caracteristiques par diagnostic', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================================================
# 2. PRETRAITEMENT DES DONNEES
# ============================================================================

print("\n2. PRETRAITEMENT DES DONNEES")
print("-"*50)

print("Valeurs uniques dans la colonne 'diagnosis' :")
print(f"   {df['diagnosis'].unique()}")

le = LabelEncoder()
df['diagnosis_encoded'] = le.fit_transform(df['diagnosis'])
print(f"\nEncodage de la variable cible :")
print(f"   B (Benin) -> {le.transform(['B'])[0]}")
print(f"   M (Malin) -> {le.transform(['M'])[0]}")

X = df.drop(['diagnosis', 'diagnosis_encoded'], axis=1)
y = df['diagnosis_encoded']

print(f"\nDimensions des features (X) : {X.shape}")
print(f"Dimensions de la cible (y) : {y.shape}")

# ============================================================================
# 3. DIVISION TRAIN/TEST
# ============================================================================

print("\n3. DIVISION TRAIN/TEST")
print("-"*50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Taille des ensembles :")
print(f"   Entrainement : {len(X_train)} patients ({len(X_train)/len(df)*100:.0f}%)")
print(f"   Test         : {len(X_test)} patients ({len(X_test)/len(df)*100:.0f}%)")

print(f"\nDistribution des classes dans les ensembles :")
print(f"   Entrainement - Classe 1 (Malin) : {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"   Test         - Classe 1 (Malin) : {y_test.sum()} ({y_test.mean()*100:.1f}%)")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nNormalisation effectuee avec StandardScaler")

# ============================================================================
# 4. REGRESSION LOGISTIQUE
# ============================================================================

print("\n4. REGRESSION LOGISTIQUE")
print("-"*50)

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print(f"Performances de la Regression Logistique :")
print(f"   Accuracy  : {accuracy_lr:.4f} ({accuracy_lr*100:.2f}%)")
print(f"   Precision : {precision_lr:.4f}")
print(f"   Recall    : {recall_lr:.4f}")
print(f"   F1-Score  : {f1_lr:.4f}")

cm_lr = confusion_matrix(y_test, y_pred_lr)
print(f"\nMatrice de confusion :")
print(f"                 Predi")
print(f"              Benin   Malin")
print(f"   Reel Benin   {cm_lr[0,0]:3d}     {cm_lr[0,1]:3d}")
print(f"   Reel Malin   {cm_lr[1,0]:3d}     {cm_lr[1,1]:3d}")

# ============================================================================
# 5. K PLUS PROCHE VOISINS (KNN)
# ============================================================================

print("\n5. K PLUS PROCHE VOISINS (KNN)")
print("-"*50)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print(f"Performances du KNN (k=5) :")
print(f"   Accuracy  : {accuracy_knn:.4f} ({accuracy_knn*100:.2f}%)")
print(f"   Precision : {precision_knn:.4f}")
print(f"   Recall    : {recall_knn:.4f}")
print(f"   F1-Score  : {f1_knn:.4f}")

cm_knn = confusion_matrix(y_test, y_pred_knn)
print(f"\nMatrice de confusion :")
print(f"                 Predi")
print(f"              Benin   Malin")
print(f"   Reel Benin   {cm_knn[0,0]:3d}     {cm_knn[0,1]:3d}")
print(f"   Reel Malin   {cm_knn[1,0]:3d}     {cm_knn[1,1]:3d}")

k_values = range(1, 21)
knn_scores = []
for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    y_pred_temp = knn_temp.predict(X_test_scaled)
    knn_scores.append(accuracy_score(y_test, y_pred_temp))

best_k = k_values[np.argmax(knn_scores)]
print(f"\nMeilleur k trouve : {best_k} (Accuracy = {max(knn_scores):.4f})")

# ============================================================================
# 6. RANDOM FOREST
# ============================================================================

print("\n6. RANDOM FOREST")
print("-"*50)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print(f"Performances du Random Forest :")
print(f"   Accuracy  : {accuracy_rf:.4f} ({accuracy_rf*100:.2f}%)")
print(f"   Precision : {precision_rf:.4f}")
print(f"   Recall    : {recall_rf:.4f}")
print(f"   F1-Score  : {f1_rf:.4f}")

cm_rf = confusion_matrix(y_test, y_pred_rf)
print(f"\nMatrice de confusion :")
print(f"                 Predi")
print(f"              Benin   Malin")
print(f"   Reel Benin   {cm_rf[0,0]:3d}     {cm_rf[0,1]:3d}")
print(f"   Reel Malin   {cm_rf[1,0]:3d}     {cm_rf[1,1]:3d}")

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 des features les plus importantes :")
for i, row in feature_importance.head(10).iterrows():
    print(f"   {row['Feature']:25s} : {row['Importance']:.4f}")

# ============================================================================
# 7. SVM (SUPPORT VECTOR MACHINE)
# ============================================================================

print("\n7. SVM (SUPPORT VECTOR MACHINE)")
print("-"*50)

svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_proba_svm = svm.predict_proba(X_test_scaled)[:, 1]

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print(f"Performances du SVM (noyau RBF) :")
print(f"   Accuracy  : {accuracy_svm:.4f} ({accuracy_svm*100:.2f}%)")
print(f"   Precision : {precision_svm:.4f}")
print(f"   Recall    : {recall_svm:.4f}")
print(f"   F1-Score  : {f1_svm:.4f}")

cm_svm = confusion_matrix(y_test, y_pred_svm)
print(f"\nMatrice de confusion :")
print(f"                 Predi")
print(f"              Benin   Malin")
print(f"   Reel Benin   {cm_svm[0,0]:3d}     {cm_svm[0,1]:3d}")
print(f"   Reel Malin   {cm_svm[1,0]:3d}     {cm_svm[1,1]:3d}")

# ============================================================================
# 8. COMPARAISON DES MODELES
# ============================================================================

print("\n8. COMPARAISON DES MODELES")
print("-"*50)

comparison_df = pd.DataFrame({
    'Modele': ['Regression Logistique', 'KNN (k=5)', 'Random Forest', 'SVM (RBF)'],
    'Accuracy': [accuracy_lr, accuracy_knn, accuracy_rf, accuracy_svm],
    'Precision': [precision_lr, precision_knn, precision_rf, precision_svm],
    'Recall': [recall_lr, recall_knn, recall_rf, recall_svm],
    'F1-Score': [f1_lr, f1_knn, f1_rf, f1_svm]
})

print(comparison_df.to_string(index=False))

best_model_idx = comparison_df['F1-Score'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Modele']
best_f1 = comparison_df.loc[best_model_idx, 'F1-Score']

print(f"\nMEILLEUR MODELE : {best_model_name}")
print(f"   F1-Score : {best_f1:.4f} ({best_f1*100:.2f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

ax1 = axes[0, 0]
models = comparison_df['Modele']
x = np.arange(len(models))
width = 0.2
ax1.bar(x - 1.5*width, comparison_df['Accuracy'], width, label='Accuracy', color='#3498DB')
ax1.bar(x - 0.5*width, comparison_df['Precision'], width, label='Precision', color='#2ECC71')
ax1.bar(x + 0.5*width, comparison_df['Recall'], width, label='Recall', color='#E74C3C')
ax1.bar(x + 1.5*width, comparison_df['F1-Score'], width, label='F1-Score', color='#9B59B6')
ax1.set_xlabel('Modele', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Comparaison des Performances par Modele', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15, ha='right')
ax1.legend()
ax1.set_ylim(0.9, 1.02)
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[0, 1]
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Benin', 'Malin'],
            yticklabels=['Benin', 'Malin'])
ax2.set_title('Regression Logistique', fontsize=12, fontweight='bold')

ax3 = axes[1, 0]
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', ax=ax3,
            xticklabels=['Benin', 'Malin'],
            yticklabels=['Benin', 'Malin'])
ax3.set_title('KNN (k=5)', fontsize=12, fontweight='bold')

ax4 = axes[1, 1]
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['Benin', 'Malin'],
            yticklabels=['Benin', 'Malin'])
ax4.set_title('Random Forest', fontsize=12, fontweight='bold')

plt.suptitle('Matrices de Confusion des 4 Modeles', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

fig2, ax5 = plt.subplots(figsize=(10, 6))
x = np.arange(len(models))
bars = ax5.bar(x, comparison_df['Accuracy'], color=['#3498DB', '#2ECC71', '#E74C3C', '#9B59B6'])
ax5.set_xticks(x)
ax5.set_xticklabels(models, rotation=15, ha='right')
ax5.set_ylabel('Accuracy', fontsize=12)
ax5.set_title('Accuracy par Modele', fontsize=14, fontweight='bold')
ax5.set_ylim(0.94, 1.0)
for bar, acc in zip(bars, comparison_df['Accuracy']):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================================
# 9. VALIDATION CROISEE
# ============================================================================

print("\n9. VALIDATION CROISEE (10-fold)")
print("-"*50)

cv_models = [log_reg, knn, rf, svm]
cv_names = ['Regression Logistique', 'KNN', 'Random Forest', 'SVM']
cv_results = []

for model, name in zip(cv_models, cv_names):
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring='f1')
    cv_results.append({
        'Modele': name,
        'F1-score moyen': cv_scores.mean(),
        'Ecart-type': cv_scores.std()
    })

cv_df = pd.DataFrame(cv_results)
print(cv_df.to_string(index=False))

# ============================================================================
# 10. CONCLUSION
# ============================================================================

print("\n" + "="*70)
print("CONCLUSION - CLASSIFICATION DU CANCER DU SEIN")
print("="*70)

print(f"""
RESUME DES RESULTATS :

   REGRESSION LOGISTIQUE :
   • Accuracy : {accuracy_lr:.2%}
   • F1-Score : {f1_lr:.4f}
   • Points forts : Interpretable, rapide, bonnes performances
   • Points faibles : Suppose une relation lineaire

   KNN (k=5) :
   • Accuracy : {accuracy_knn:.2%}
   • F1-Score : {f1_knn:.4f}
   • Points forts : Simple, non parametrique
   • Points faibles : Sensible a l'echelle, lent en prediction

   RANDOM FOREST :
   • Accuracy : {accuracy_rf:.2%}
   • F1-Score : {f1_rf:.4f}
   • Points forts : Robuste, non lineaire, importance des features
   • Points faibles : Moins interpretable, plus lent

   SVM (noyau RBF) :
   • Accuracy : {accuracy_svm:.2%}
   • F1-Score : {f1_svm:.4f}
   • Points forts : Bonne generalisation, noyau non lineaire
   • Points faibles : Sensible aux hyperparametres

MEILLEUR MODELE : {best_model_name}
   • F1-Score : {best_f1:.4f} ({best_f1*100:.2f}%)
   • Ce modele est recommande pour la prediction du cancer du sein

ANALYSE :
   1. Tous les modeles montrent d'excellentes performances (accuracy > 95%)
   2. Le {best_model_name} est le plus performant pour ce jeu de donnees
   3. Le Random Forest offre l'avantage supplementaire de l'importance des features
   4. La Regression Logistique est recommandee si l'interpretabilite est prioritaire
""")